In [ ]:
import os, shutil
from datetime import datetime
import numpy as np
import pandas as pd
import geopandas as gpd

SOURCE_PATH = "../Grids/halfres_frames.gpkg"      # your blocked tile grid
LABELLING_PATH = "../Grids/labelling.gpkg"        # the file you'll label in QGIS
SEED = 1
MIN_VALID_FRACTION = 0.9
TEST_BLOCKS = {"KD02": ["2002", "4002"], "KD06": ["1002", "2005"]}
SOURCE_LAYERS = {"KD02": "KD02_frames", "KD06": "KD06_frames"}


In [ ]:
forest_tables = []
for forest, layer_name in SOURCE_LAYERS.items():
    grid = gpd.read_file(SOURCE_PATH, layer=layer_name)
    is_usable = (grid["block_id"].notna() & ~grid["is_buffer"] & ~grid["edge_excluded"]
                 & (grid["valid_frac"] >= MIN_VALID_FRACTION))
    usable = grid[is_usable].copy()
    usable["forest"] = forest
    usable["tile_id"] = forest + "_" + usable["frame_id"]
    usable["block"] = forest + "_" + usable["block_id"]   # block IDs repeat between forests
    forest_tables.append(usable)
    print(f"{forest}: {len(grid)} tiles in grid, {len(usable)} usable")

tiles = pd.concat(forest_tables, ignore_index=True)

tiles["role"] = "train"
for forest, test_block_ids in TEST_BLOCKS.items():
    is_test = (tiles["forest"] == forest) & tiles["block_id"].isin(test_block_ids)
    tiles.loc[is_test, "role"] = "test"


In [ ]:
tiles.groupby(["forest", "role"]).agg(tiles=("tile_id", "size"),
                                     blocks=("block", "nunique"))

random_generator = np.random.default_rng(SEED)
tiles = tiles.sort_values("tile_id").reset_index(drop=True)   # fixed start, so the seed is reproducible
tiles["random_draw"] = random_generator.random(len(tiles))

rank_in_block = tiles.groupby("block")["random_draw"].rank()
block_size = tiles.groupby("block")["random_draw"].transform("size")
tiles["position_in_block"] = rank_in_block / block_size        # 0–1 within each block

tiles = tiles.sort_values(["forest", "role", "position_in_block", "tile_id"])
tiles["order"] = tiles.groupby(["forest", "role"]).cumcount() + 1


In [ ]:
first_batch = tiles[(tiles["forest"] == "KD02") & (tiles["role"] == "train")
                    & (tiles["order"] <= 500)]
first_batch.groupby("block_id").size()

In [ ]:
#RUN ONCE, SAVES TILES TO GPKG
for column in ["status", "condition", "notes"]:
    tiles[column] = pd.Series([None] * len(tiles), dtype="object", index=tiles.index)
tiles["batch"] = pd.array([pd.NA] * len(tiles), dtype="Int64")

columns_to_keep = ["tile_id", "frame_id", "forest", "block", "block_id", "role", "order",
                   "batch", "status", "condition", "notes", "row", "col", "valid_frac",
                   "brightness", "exg", "glcm_contrast", "geometry"]
labelling_tiles = gpd.GeoDataFrame(tiles[columns_to_keep], geometry="geometry")
labelling_tiles = labelling_tiles.set_crs("EPSG:27700", allow_override=True)

if os.path.exists(LABELLING_PATH):
    print(f"{LABELLING_PATH} already exists. Not overwriting it.")
else:
    labelling_tiles.to_file(LABELLING_PATH, layer="tiles", driver="GPKG")
    print(f"Saved {len(labelling_tiles)} tiles to {LABELLING_PATH}")


In [ ]:
FOREST, ROLE, BATCH_SIZE = "KD02", "train", 200

labelling_tiles = gpd.read_file(LABELLING_PATH, layer="tiles")
labelling_tiles["batch"] = labelling_tiles["batch"].astype("Int64")
backup_path = f"{LABELLING_PATH}.{datetime.now():%Y%m%d-%H%M%S}.bak"
shutil.copy2(LABELLING_PATH, backup_path)

unassigned = labelling_tiles[(labelling_tiles["forest"] == FOREST)
                             & (labelling_tiles["role"] == ROLE)
                             & labelling_tiles["batch"].isna()]
next_tiles = unassigned.sort_values("order").head(BATCH_SIZE).index

if labelling_tiles["batch"].notna().any():
    batch_number = int(labelling_tiles["batch"].max()) + 1
else:
    batch_number = 1

labelling_tiles.loc[next_tiles, "batch"] = batch_number
labelling_tiles.to_file(LABELLING_PATH, layer="tiles", driver="GPKG")
print(f'Batch {batch_number}: {len(next_tiles)} tiles. In QGIS filter with:  "batch" = {batch_number}')


In [ ]:
labelling_tiles = gpd.read_file(LABELLING_PATH, layer="tiles")
progress = labelling_tiles.assign(
    assigned=labelling_tiles["batch"].notna(),
    labelled=labelling_tiles["status"].isin(["done", "unsure", "skip"]),
    skipped=labelling_tiles["status"].eq("skip"),
)
progress.groupby(["forest", "role"])[["assigned", "labelled", "skipped"]].sum()
